<a href="https://colab.research.google.com/github/Gustavo-Michael/Mineria_178318/blob/main/Proyecto_Final_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto final - Inteligencia Artificial 2 - T58A

Integrantes:

*   Daniel Gil González - 178396
*   Daniel Alejandro Venegas Rivera - 177603
*   Pablo de Jesús Salazar Hernández - 171580
*   Leonardo Serrano Zermeño- 177301
*   Gustavo Michael Larios Guerra - 178318

En este proyecto final se busca crear un programa que pueda entrar a una página de noticias, extraer su contenido y analizarlo automáticamente. La idea principal es hacer web scraping para obtener varias noticias y luego buscar dentro de ellas una palabra clave que nosotros elijamos como en este caso, UPSLP.

Si la palabra aparece en alguna noticia, el programa va a mostrar el título y además generar un pequeño resumen del contenido. Con esto ponemos en práctica técnicas de manejo de datos, procesamiento de texto y generación automática de resúmenes.


In [ ]:
#Instalamos las librerías necesarias para poder hacer peticiones a la página de noticias y para analizar el contenido HTML que obtenemos
!pip install requests beautifulsoup4

In [ ]:
#Instalamos la librería transformers junto con el soporte para pytorch, esto es necesario para poder cargar y usar el modelo que va a generar los resúmenes
!pip install "transformers[torch]"

In [ ]:
#Instalamos Playwright, una herramienta que nos ayuda a automatizar la navegación web, sirve especialmente cuando la página de noticias carga contenido dinámico y no podemos obtener todo solo con requests
!pip install playwright
#Instalamos el navegador Chromium para que playwright pueda usarlo al momento de hacer scraping
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 MB 17.5 MB/s eta 0:00:00
173.9 MiB [] 0% 0.0s173.9 MiB [] 0% 9.8s173.9 MiB [] 0% 6.3s173.9 MiB [] 1% 4.6s173.9 MiB [] 1% 3.7s173.9 MiB [] 2% 3.7s173.9 MiB [] 3% 3.0s173.9 MiB [] 3% 2.7s173.9 MiB [] 4% 2.6s173.9 MiB [] 5% 2.6s173.9 MiB [] 5% 2.7s173.9 MiB [] 5% 3.0s173.9 MiB [] 6% 2.8s173.9 MiB [] 7% 2.7s173.9 MiB [] 8% 2.6s173.9 MiB [] 9% 2.5s173.9 MiB [] 10% 2.5s173.9 MiB [] 11% 2.4s173.9 MiB [] 12% 2.3s173.9 MiB [] 13% 2.3s173.9 MiB [] 13% 2.4s173.9 MiB [] 14% 2.4s173.9 MiB [] 15% 2.3s173.9 MiB [] 16% 2.2s173.9 MiB [] 17% 2.2s173.9 MiB [] 18% 2.1s173.9 MiB [] 19% 2.1s173.9 MiB [] 20% 2.0s173.9 MiB [] 21% 2.0s173.9 MiB [] 22% 2.0s173.9 MiB [] 23% 2.0s173.9 MiB [] 24% 1.9s173.9 MiB [] 25% 1.9s173.9 MiB [] 25% 1.8s173.9 MiB [] 26% 1.8s173.9 MiB [] 26% 1.9s173.9 MiB [] 27% 1.9s173.9 MiB [] 28% 1.9s173.9 MiB [] 29% 1.8s173.9 MiB [] 30% 1.8s173.9 MiB [] 31% 1.7s173.9 MiB [] 32% 1.7s173.9 MiB [] 33% 1.7s173.9 MiB [] 34% 1.6s173.9 MiB [] 

In [ ]:
#Instalamos las dependencias del sistema que necesita playwright
!playwright install-deps

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main

In [ ]:
#Importamos requests para hacer peticiones HTTP a las páginas de noticias
import requests
#BeautifulSoup nos sirve para analizar y extraer información del html
from bs4 import BeautifulSoup
#Usamos re para hacer búsquedas con expresiones regulares al limpiar texto
import re
#Playwright se usa para interactuar con páginas dinámicas.
import asyncio
from playwright.async_api import async_playwright
#Torch se usa como base para cargar y manejar modelos de IA
import torch
#Importamos el tokenizador y el modelo encoder-decoder para generar resúmenes de las noticias
from transformers import RobertaTokenizerFast, EncoderDecoderModel

In [ ]:
#Definimos nuestras variables de busqueda
#Palabra clave que vamos a buscar dentro de las noticias
key_word = "UPSLP"
#URL base de la página donde se hace la busqueda aquí se añade la palabra clave en minusculas y luego se le agrega el número de página
url_target = "https://oem.com.mx/elsoldesanluis/buscar/?q=" + key_word.lower() + "&page="
#Headers para que la página no bloquee nuestras requests, simulando que estamos entrando desde un navegador normal
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

In [ ]:
#Funcion que se usa para hacer las requests
def obtener_sopa(url):
  try:
#Hacemos la request a la página usando los headers para evitar bloqueos
        respuesta = requests.get(url, headers=headers)
#Si la página responde con un error, esto lo detecta y detiene el proceso
        respuesta.raise_for_status()
#Convertimos el html en un objeto beautifulsoup para poder analizarlo
        return BeautifulSoup(respuesta.text, 'html.parser')
  except Exception as e:
#Si algo falla (como que la página no exista o haya un error de conexión) se manda un aviso y devolvemos "None" para que el programa no se rompa
        print(f"Error al conectar con {url}: {e}")
        return None

# Busqueda mediante metadatos
## buscar_noticias_metadata(url_base, key_word, cont)
Esta función busca títulos de noticias que contengan una palabra clave específica en una página web. Utiliza requests y BeautifulSoup para analizar el contenido HTML de la página de forma rápida.
- **Parámetros:**

1. url_base: La URL de la página web donde se realizará la búsqueda.
2. key_word: La palabra clave que se desea buscar en los titulares de las noticias.
3. cont: Contador.

In [ ]:
def buscar_noticias_metadata(url_base, key_word, cont):
    soup = obtener_sopa(url_base) # Obtiene el HTML de la url_base usando la función obtener_sopa
    if not soup:
        return []

    noticias_candidatas = []
    titulares = soup.find_all(['h1', 'h2', 'h3', 'h4']) # Busca todos los elementos HTML que contienen titularers.

    for titular in titulares:
        texto_titular = titular.get_text(strip=True) # Verifica si el titular contiene la key_word

        if re.search(key_word, texto_titular, re.IGNORECASE):
            enlace_padre = titular.find_parent('a') or titular.find('a') # Busca la etiqueta <a> que contiene hipervínculos en HTML

            if enlace_padre and 'href' in enlace_padre.attrs:
                url_noticia = enlace_padre['href'] # Extrae el enlace de la noticia
                if not url_noticia.startswith('http'):
                    url_noticia = "https://oem.com.mx" + url_noticia # Ajusta el dominio base según corresponda

                # Evitar duplicados en la misma iteración
                if any(n["url"] == url_noticia for n in noticias_candidatas):
                    continue

                # Almacena el título y la URL de las noticias coincidentes
                noticias_candidatas.append({
                    'titulo': texto_titular,
                    'url': url_noticia,
                    'contenido': None # Se llena en procesar_lote_noticias
                })

    return noticias_candidatas # Regresa las noticias y el URL

## procesar_lote_noticias(lista_noticias)

Esta función extrae el contenido completo de las noticias que fueron identificadas por buscar_noticias_metadata. Utiliza Playwright para navegar a cada URL y extraer el texto principal
- **Parámetros:**

1. lista_noticias: Una lista de diccionarios, donde cada diccionario contiene el titulo y la url de una noticia (la salida de buscar_noticias_metadata).

In [ ]:
async def procesar_lote_noticias(lista_noticias):
    if not lista_noticias:
        return []

    print(f"--- Iniciando extracción masiva de {len(lista_noticias)} noticias ---")

    async with async_playwright() as p:
        # Inicia una instancia de navegador Chromium usando Playwright UNA sola vez para todo el lote.
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()

        # Semáforo para limitar pestañas simultáneas (evita que crashee Colab)
        sem = asyncio.Semaphore(5)

        # Procesa cada noticia individualmente
        async def tarea_extraccion(noticia):
            async with sem: # Solo permite 5 ejecuciones simultáneas de este bloque
                page = await context.new_page() #Abre una nueva página del navegador
                try:
                    # Bloqueo de recursos para velocidad extrema
                    await page.route("**/*", lambda route: route.abort()
                        if route.request.resource_type in ["image", "media", "font", "stylesheet"]
                        else route.continue_()
                    )

                    # Navega a la url de la noticia
                    await page.goto(noticia['url'], timeout=20000, wait_until="domcontentloaded")

                    # Script de extracción rápida (JS en navegador)
                    # Busca el #ev-content y saca los p largos
                    contenido = await page.evaluate("""() => {
                        const contenedor = document.querySelector('#ev-content');
                        if (!contenedor) return null;

                        const paragraphs = Array.from(contenedor.querySelectorAll('p'));
                        const textos = paragraphs
                            .map(p => p.innerText.trim())
                            .filter(t => t.length > 90);

                        return textos.join('\\n');
                    }""")

                    print(f"Procesado: {noticia['titulo']}...")
                    print("\n")
                    print("=" *50)
                    print(contenido)

                    noticia['contenido'] = contenido if contenido else "Sin contenido válido detectado."
                    #print(f"Procesado: {noticia['titulo']}...")

                except Exception as e:
                    print(f"Error en {noticia['url']}: {e}")
                    noticia['contenido'] = f"Error: {e}"
                finally:
                    await page.close() # Cierra la página una vez que se terminó de procesar la noticia

        # Creamos las tareas y las ejecutamos
        tareas = [tarea_extraccion(n) for n in lista_noticias] # Crea una lista de estas tareas de extracción para todas las noticias en lista_noticias
        await asyncio.gather(*tareas) # Ejecuta todas las tareas en paralelo usando asyncio.gather

        await browser.close() # Cierra el navegador

    return lista_noticias # Devuelve la lista de noticias con el conte

In [ ]:
async def buscar_todo2(url_target_base, key_word): #itera através de las paginas de resultados
    memoria_total = []
    index = 0
    url_actual = url_target_base # Guardar base para no corromper string

    while True:
        # Nota: Ajusté la lógica del URL porque concatenar strings al final
        # en bucle suele generar urls invalidas tipo url0123.
        url_paginada = f"{url_target_base}{index}"

        print(f"\nBuscando en página {index}: {url_paginada}")

        # 1. Obtener candidatos (Rápido, requests/BS4)
        candidatos = buscar_noticias_metadata(url_paginada, key_word, len(memoria_total))

        if not candidatos:
            print("No se encontraron más noticias en esta página.")
            # Condición de salida: si no hay noticias nuevas, rompemos el bucle
            if index > 0:
                 break

        # Filtrar duplicados que ya tengamos en memoria_total
        urls_existentes = set(m['url'] for m in memoria_total)
        nuevos_candidatos = [c for c in candidatos if c['url'] not in urls_existentes]

        if not nuevos_candidatos:
            print("Todas las noticias de esta página ya estaban guardadas.")
            index += 1
            continue

        # 2. Extraer contenido MASIVAMENTE (Rápido, Playwright asíncrono)
        lote_procesado = await procesar_lote_noticias(nuevos_candidatos)

        memoria_total.extend(lote_procesado)

        print(f"Total noticias acumuladas: {len(memoria_total)}")

        # Tu lógica de salida original
        # (Si la página no trajo nada nuevo comparado con lo anterior, salir)
        if len(nuevos_candidatos) == 0:
            break

        index += 1

        # Safety break para pruebas (quitar en producción)
        if index > 5: break
    return memoria_total

---

# Funciones de Resumen de Texto

---

generate_summary(text)

Esta funcion genera un resumen automatico de un texto utilizando un modelo de IA especializado en español.

### Que para parametros usa?

text: Texto completo que se desea resumir de tipo string

### Proceso:

Tokeniza el texto de entrada (convierte a formato que entiende el modelo)
Prepara los tensores para el dispositivo (GPU/CPU)
Genera el resumen usando el modelo pre-entrenado
Decodifica el resultado para obtener texto legible
Retorno: Resumen conciso del texto original como string.


In [ ]:
# Configuracion del dispositivo (GPU si esta disponible, sino CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Ruta del modelo preentrenado para resumir texto en español
ckpt = 'Narrativa/bsc_roberta2roberta_shared-spanish-finetuned-mlsum-summarization'
# Cargamos el tokenizador especifico para el modelo
tokenizer = RobertaTokenizerFast.from_pretrained(ckpt)
# Cargamos el modelo y lo movemos al dispositivo configurado (GPU/CPU)
model = EncoderDecoderModel.from_pretrained(ckpt).to(device)

def generate_summary(text):
    # Tokenizamos el texto de entrada: convertimos el texto en tokens numericos que el modelo entiende
    inputs = tokenizer([text], padding="max_length", truncation=True, max_length=512, return_tensors="pt")

    # Movemos los IDs de los tokens al dispositivo adecuado (GPU/CPU) para acelerar el procesamiento
    input_ids = inputs.input_ids.to(device)

    # Movemos la mascara de atención al mismo dispositivo
    # Esta mascara indica al modelo que tokens son relevantes (1) y cuales son padding (0)
    attention_mask = inputs.attention_mask.to(device)

    # Generamos el resumen usando el modelo:
    # El modelo procesa la entrada y genera una secuencia de tokens que forman el resumen
    output = model.generate(input_ids, attention_mask=attention_mask)

    # Decodificamos los tokens generados a texto legible
    # skip_special_tokens=True: elimina tokens especiales como <s>, </s>, <pad>, etc.
    # output[0] porque model.generate devuelve un lote, y aqui solo procesamos un texto
    return tokenizer.decode(output[0], skip_special_tokens=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/615M [00:00<?, ?B/s]

The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']


model.safetensors:   0%|          | 0.00/615M [00:00<?, ?B/s]

The following encoder weights were not tied to the decoder ['roberta/pooler']


resumir_todo(lista_noticias)
Esta funcion procesa y muestra resumenes de una lista de noticias

### Que parametros usa?

lista_noticias: Lista de diccionarios con las claves 'titulo', 'url' y 'contenido'

### Comportamiento:

- Verifica si hay noticias para procesar
- Muestra un encabezado visual con separadores
- Para cada noticia:
  - Imprime título y URL
  - Genera y muestra el resumen del contenido
  - Agrega un separador visual entre noticias
- Muestra el total de noticias procesadas

Nota: Esta funcion no retorna ningun valor, solo imprime resultados en la consola.

In [ ]:
def resumir_todo(lista_noticias):
    # Verificamos si la lista de noticias esta vacia
    if not lista_noticias:
        # Si no hay noticias, mostramos un mensaje y terminamos la funcion
        print("No se encontraron noticias.")
        return

    print("\n" + "="*50)
    print("Mostrando resumenes")
    print("="*50)

    # Iteramos sobre cada noticia en la lista
    for noticia in lista_noticias:
        # Imprimimos el título y URL de la noticia actual para contexto
        # Formato para el Titulo y el URL
        print(f"Título: {noticia['titulo']} URL: {noticia['url']}")

        # Llamamos a generate_summary() para crear un resumen del contenido de la noticia
        # e imprimimos directamente el resultado
        print(generate_summary(noticia['contenido']))
        print("-" * 50)
    # Al finalizar, mostramos un resumen estadístico: cuántas noticias se procesaron en total
    print(f"Se resumieron {len(lista_noticias)} noticias.")

---

Realizamos la busqueda y guardamos los resultados

In [ ]:
lista_resultados = await buscar_todo2(url_target, key_word)
#Se ejecuta la función principal de búsqueda de manera asíncrona


Buscando en página 0: https://oem.com.mx/elsoldesanluis/buscar/?q=upslp&page=0
--- Iniciando extracción masiva de 7 noticias ---
Procesado: UPSLP anuncia inversión en estrategia de Seguridad...


La Universidad Politécnica de San Luis Potosí (UPSLP) anunció una inversión de dos millones de pesos destinada a reforzar la seguridad dentro y fuera de su campus, con el objetivo de garantizar un entorno confiable y protegido para su comunidad universitaria.
El rector de la institución, Néstor Eduardo Garza Álvarez, informó que los recursos se utilizarán para fortalecer la infraestructura tecnológica de vigilancia y mejorar los mecanismos de respuesta ante cualquier eventualidad “nuestra prioridad es la seguridad de nuestra comunidad. Queremos que cada estudiante, docente y colaborador se desarrolle en un entorno protegido, sin puntos ciegos y con monitoreo permanente”, declaró.
El proyecto contempla la instalación de más de un centenar de cámaras de videovigilancia, la creación de un Centro

Hacemos un resumen de todas las noticias que se encontraron

In [ ]:
resumir_todo(lista_resultados)
#se pasa la lista completa de noticias con su contenido al modelo de IA
#se imprimirá el titulo, la URL y el resúmen generado para cada articulo.


Mostrando resumenes
Título: UPSLP anuncia inversión en estrategia de Seguridad URL: https://oem.com.mx/elsoldesanluis/local/upslp-anuncia-inversion-en-estrategia-de-seguridad-26730051
La Universidad Politécnica de San Luis Potosí construirá un centenar de cámaras de videovigilancia en el campus universitario
--------------------------------------------------
Título: Estudiantes de Mercadotecnia Internacional de la UPSLP presentan “Sales Force 2025” URL: https://oem.com.mx/elsoldesanluis/local/estudiantes-de-mercadotecnia-internacional-de-la-upslp-presentan-sales-force-2025-26636385
Los alumnos de la licenciatura en Mercadotecnia Internacional de San Luis Potosí presentan el evento académico 'Sales Force 2025'
--------------------------------------------------
Título: Video genera controversia en la UPSLP; rector niega represión contra estudiantes URL: https://oem.com.mx/elsoldesanluis/local/video-genera-controversia-en-la-upslp-rector-niega-represion-contra-estudiantes-25420578
El rec